In [80]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import sys
import os
sys.path.append(os.path.dirname(os.path.abspath('.')))

from sqlalchemy import text
from db.database import engine
from etl.constants import EUROPEAN_COUNTRIES, DECOUPLING_START_YEAR, DECOUPLING_END_YEAR

BASE_YEARS  = [DECOUPLING_START_YEAR, 2000]

In [81]:
query = """
    SELECT
        c.name AS country,
        c.iso_code,
        e.year,
        e.co2_total,
        e.gdp
    FROM emissions e
    JOIN countries c ON c.id = e.country_id
    WHERE e.year BETWEEN :start AND :end
    ORDER BY c.iso_code, e.year
"""

with engine.connect() as conn:
    df = pd.read_sql(
        text(query),
        conn,
        params={'start': BASE_YEARS[0], 'end': DECOUPLING_END_YEAR}
    )

df_europe = df[df['iso_code'].isin(EUROPEAN_COUNTRIES)].copy()

print(f"European dataset: {len(df_europe)} rows, {df_europe['iso_code'].nunique()} countries")

2026-06-11 20:18:32,008 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-11 20:18:32,012 INFO sqlalchemy.engine.Engine SELECT pg_catalog.pg_class.relname 
FROM pg_catalog.pg_class JOIN pg_catalog.pg_namespace ON pg_catalog.pg_namespace.oid = pg_catalog.pg_class.relnamespace 
WHERE pg_catalog.pg_class.relname = %(table_name)s AND pg_catalog.pg_class.relkind = ANY (ARRAY[%(param_1)s, %(param_2)s, %(param_3)s, %(param_4)s, %(param_5)s]) AND pg_catalog.pg_table_is_visible(pg_catalog.pg_class.oid) AND pg_catalog.pg_namespace.nspname != %(nspname_1)s
2026-06-11 20:18:32,016 INFO sqlalchemy.engine.Engine [cached since 1.024e+04s ago] {'table_name': <sqlalchemy.sql.elements.TextClause object at 0x0000012323C7F5D0>, 'param_1': 'r', 'param_2': 'p', 'param_3': 'f', 'param_4': 'v', 'param_5': 'm', 'nspname_1': 'pg_catalog'}
2026-06-11 20:18:32,026 INFO sqlalchemy.engine.Engine 
    SELECT
        c.name AS country,
        c.iso_code,
        e.year,
        e.co2_total,
        e.gdp
    FR

2026-06-11 20:18:32,162 INFO sqlalchemy.engine.Engine ROLLBACK
European dataset: 1156 rows, 34 countries


In [82]:
def build_index(df: pd.DataFrame, base_year: int = DECOUPLING_START_YEAR) -> pd.DataFrame:
    df = df.copy()

    results = []
    for _, group in df.groupby('iso_code'):
        base = group[group['year'] == base_year]
        if base.empty:
            continue

        base_gdp  = base['gdp'].values[0]
        base_co2  = base['co2_total'].values[0]

        if pd.isna(base_gdp) or pd.isna(base_co2):
            continue

        group = group.copy()
        group['gdp_index'] = group['gdp'] / base_gdp * 100
        group['co2_index'] = group['co2_total'] / base_co2 * 100
        results.append(group)

    return pd.concat(results, ignore_index=True)


df_indexed_1990 = build_index(df_europe, BASE_YEARS[0])
df_indexed_2000 = build_index(df_europe, BASE_YEARS[1])

print(f"Indexed (1990 base): {len(df_indexed_1990)} rows, {df_indexed_1990['iso_code'].nunique()} countries")
print("\nSample - United Kingdom (1990 base):")
print(df_indexed_1990[df_indexed_1990['iso_code'] == 'GBR'][
    ['year', 'gdp_index', 'co2_index']
].query('year in [1990, 2000, 2010, 2020]').round(1).to_string(index=False))

Indexed (1990 base): 1156 rows, 34 countries

Sample - United Kingdom (1990 base):
 year  gdp_index  co2_index
 1990      100.0      100.0
 2000      125.5       94.5
 2010      145.5       85.0
 2020      155.1       54.2


In [ ]:
# Decoupling elasticity = (% change in CO2) / (% change in GDP) over the period
CATEGORY_ORDER = [
    'Strong decoupling', 
    'Weak decoupling', 
    'Coupling',
    'Expansive neg. decoupling', 
    'Recessive', 
    'Strong neg. decoupling'
]
CATEGORY_COLORS = {
    'Strong decoupling':         "#01853A",
    'Weak decoupling':           '#A6D96A',
    'Coupling':                  '#FEE08B',
    'Expansive neg. decoupling': "#DF2E25",
    'Recessive':'#878787',
    'Strong neg. decoupling':    "#7F14A2"
}

def tapio_category(gdp_index, co2_index):
    delta_gdp = gdp_index - 100
    delta_co2 = co2_index - 100
    if delta_gdp > 0:
        if delta_co2 <= 0:
            return 'Strong decoupling'
        elasticity = delta_co2 / delta_gdp
        if elasticity < 0.8:  return 'Weak decoupling'
        if elasticity <= 1.2: return 'Coupling'
        return 'Expansive neg. decoupling'
    if delta_co2 > 0:
        return 'Strong neg. decoupling'
    return 'Recessive' 

def latest_indexed(df_indexed):
    valid = df_indexed.dropna(subset=['gdp_index', 'co2_index'])
    return valid.sort_values('year').groupby('iso_code').tail(1)

def decoupling_table(df_indexed):
    threshold = 1 
    snap = latest_indexed(df_indexed).copy()
    delta_gdp = snap['gdp_index'] - 100
    delta_co2 = snap['co2_index'] - 100
    snap['elasticity'] = np.where(delta_gdp.abs() < threshold, np.nan, (delta_co2 / delta_gdp))
    snap['category'] = [tapio_category(gdp, co2) for gdp, co2 in zip(snap['gdp_index'], snap['co2_index'])]
    return snap.sort_values('elasticity')

def elasticity_bar(decoupling_df, base_year):
    chart_df = decoupling_df.dropna(subset=['elasticity'])
    fig = px.bar(
        chart_df, 
        x='country', 
        y='elasticity', 
        color='category',
        category_orders={'category': CATEGORY_ORDER}, 
        color_discrete_map=CATEGORY_COLORS,
        title=f'Decoupling elasticity index (base {base_year})<br>'
              '<sup> e&lt; 0 strong decoupling | e = 1 coupling | e &gt; 1.2 emissions outpace GDP</sup>',
        labels={
            'elasticity': 'Decoupling elasticity', 
            'country': '', 
            'category': 'Category'
        },
        height=600
    )
    fig.add_hline(y=0, line_dash='dash', line_color='green', annotation_text='decoupling (e = 0)')
    fig.add_hline(y=1, line_dash='dot', line_color='gray', annotation_text='coupling (e = 1)')
    fig.update_layout(xaxis_tickangle=-45)
    fig.show()

decoupling_1990 = decoupling_table(df_indexed_1990)
decoupling_2000 = decoupling_table(df_indexed_2000)

elasticity_bar(decoupling_table(df_indexed_1990), 1990)
elasticity_bar(decoupling_table(df_indexed_2000), 2000)

print('\nTop 10 strongest territorial decouplers (1990 base):')
print(decoupling_1990.head(10)[['country', 'gdp_index', 'co2_index', 'elasticity', 'category']].round(2).to_string(index=False))
print('\nTop 10 strongest territorial decouplers (2000 base):')
print(decoupling_2000.head(10)[['country', 'gdp_index', 'co2_index', 'elasticity', 'category']].round(2).to_string(index=False))


Top 10 strongest territorial decouplers (1990 base):
       country  gdp_index  co2_index  elasticity          category
        Latvia     121.73      33.61       -3.06 Strong decoupling
       Georgia     107.07      80.81       -2.72 Strong decoupling
       Estonia     147.79      32.42       -1.41 Strong decoupling
     Lithuania     166.23      36.08       -0.97 Strong decoupling
       Belarus     150.27      52.67       -0.94 Strong decoupling
United Kingdom     173.69      51.69       -0.66 Strong decoupling
       Finland     168.70      63.84       -0.53 Strong decoupling
      Bulgaria     174.40      61.24       -0.52 Strong decoupling
         Italy     144.77      77.61       -0.50 Strong decoupling
        Greece     163.19      69.04       -0.49 Strong decoupling

Top 10 strongest territorial decouplers (2000 base):
       country  gdp_index  co2_index  elasticity          category
        Greece     118.06      55.95       -2.44 Strong decoupling
       Ukraine     12

In [ ]:
# Decoupling map: GDP index vs CO2 index, with the Tapio zones drawn in (2000 base)
scatter_df = decoupling_2000.copy()

fig = px.scatter(
    scatter_df, 
    x='gdp_index', 
    y='co2_index', 
    color='category', 
    text='iso_code',
    category_orders={'category': CATEGORY_ORDER}, 
    color_discrete_map=CATEGORY_COLORS,
    title='Territorial decoupling map (2000 = 100)<br>'
          '<sup>Bottom-right = GDP up & CO2 down = strong decoupling. Diagonal = perfect coupling.</sup>',
    labels={
        'gdp_index': 'GDP index (2000 = 100)', 
        'co2_index': 'Territorial CO2 index (2000 = 100)',
        'category': 'Tapio category'
    },
    height=700
)
axis_lo = min(scatter_df['gdp_index'].min(), scatter_df['co2_index'].min(), 95) - 5
axis_hi = max(scatter_df['gdp_index'].max(), scatter_df['co2_index'].max(), 105) + 5
fig.add_trace(go.Scatter(x=[axis_lo, axis_hi], y=[axis_lo, axis_hi], mode='lines',
                         line=dict(dash='dot', color='black'), name='coupling (e=1)'))
fig.add_hline(y=100, line_dash='dash', line_color='gray')
fig.add_vline(x=100, line_dash='dash', line_color='gray')
fig.add_annotation(x=axis_hi- 140, 
                   y=axis_lo + 11, 
                   text='STRONG DECOUPLING',
                   showarrow=False, 
                   font=dict(color='#1a9850')
)
fig.update_traces(textposition='top center')
fig.show()

In [ ]:
print("=" * 26)
print("NOTEBOOK 09 - KEY FINDINGS")
print("=" * 26)

print("""
Metric: Tapio decoupling elasticity = (% change CO2) / (% change GDP) over the period.
   e < 0      strong decoupling (GDP up, CO2 down) - the goal
   0 .. 0.8   weak decoupling
   0.8 .. 1.2 coupling (GDP and CO2 are moving together)
   e > 1.2    emissions outpacing GDP
This is the FLATTERING story: it can't see emissions that were offshored, only territorial CO2.
""")

print("Strongest territorial decouplers (2000 base):")
print(decoupling_2000.head(6)[['country', 'elasticity', 'category']].round(2).to_string(index=False))

print("\nStrongest territorial decouplers (1990 base):")
print(decoupling_1990.head(6)[['country', 'elasticity', 'category']].round(2).to_string(index=False))

print("\nBase-year effect: under the 1990 base, post-Soviet deindustrialization inflates the ranking")
print("(Romania, Poland, the Baltics, Slovakia, Hungary) - it's mostly a collapsed economy or 'low base' effect,")
print("not an aimed decarbonization. The 2000 base partially neutralizes it and you can see it comparing two charts.")

worst = decoupling_2000.dropna(subset=['elasticity']).iloc[-1]
print(f"\nWorst performer (2000 base): {worst['country']} - {worst['category'].lower()} "
      f"(GDP and territorial CO2 both rising together).")

NOTEBOOK 09 - KEY FINDINGS

Metric: Tapio decoupling elasticity = (% change CO2) / (% change GDP) over the period.
   e < 0      strong decoupling (GDP up, CO2 down) - the goal
   0 .. 0.8   weak decoupling
   0.8 .. 1.2 coupling (GDP and CO2 are moving together)
   e > 1.2    emissions outpacing GDP
This is the FLATTERING story: it can't see emissions that were offshored, only territorial CO2.

Strongest territorial decouplers (2000 base):
       country  elasticity          category
        Greece       -2.44 Strong decoupling
       Ukraine       -2.26 Strong decoupling
         Italy       -1.89 Strong decoupling
      Portugal       -1.59 Strong decoupling
United Kingdom       -1.18 Strong decoupling
       Finland       -1.08 Strong decoupling

Strongest territorial decouplers (1990 base):
       country  elasticity          category
        Latvia       -3.06 Strong decoupling
       Georgia       -2.72 Strong decoupling
       Estonia       -1.41 Strong decoupling
     Lithuani